<a href="https://colab.research.google.com/github/Yshi-m319/TLS-Handson/blob/main/TLS%E4%BD%93%E9%A8%93_ca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 注意

- 簡便化のため、プロトコルを大胆に改変しています
- データフォーマットは独自のものを使っています
- 間違いが含まれてる可能性があります

# 🏛️ CA役（主催者）マニュアル

あなたは **CA（認証局）役** だよ。セッション全体を仕切る大切なポジションだ！

| 役割 | やること | タイミング |
|------|----------|------------|
| 🏛️ CA（あなた） | CA鍵・証明書を生成し、サーバーのCSRに署名して server.crt を発行する | セッション前 |
| 🏛️ CA（あなた） | ca.crt を全参加者に配布する（ルートCA証明書ストアの比喩） | セッション前 |
| 🖥️ サーバー | ClientHello受信 → ServerHello/Certificate送信 → PMS復号 → 鍵導出 → Finished | 当日 |
| 💻 クライアント | ClientHello送信 → 証明書検証 → PMS生成/暗号化 → 鍵導出 → Finished → 暗号文送信 | 当日 |

# 0. 前準備（セッション前にやっておこう）

## 0.1 CA秘密鍵と自己署名証明書を生成しよう！

まずはCAとしての「信頼の根拠」を作るよ。

生成するのは2つ：
- **`ca.key`** — CA秘密鍵。**絶対に公開しない！** これが漏洩すると攻撃者が偽のサーバー証明書を作り放題になる。実世界の DigiCert や Let's Encrypt が厳重に保管しているのと同じ秘密だよ。
- **`ca.crt`** — CA自己署名証明書。これはあとで全員に配布する（公開情報でOK）。

In [ ]:
# CA 秘密鍵を生成
!openssl genrsa -out ca.key 2048

In [ ]:
# CA 自己署名証明書を生成（有効期限 1日）
!openssl req -new -x509 -days 1 \
  -key ca.key -out ca.crt \
  -subj "/CN=TLS-Theater-CA"

In [ ]:
# 内容を確認してみよう（Subject と Validity が表示されれば OK！）
!openssl x509 -in ca.crt -noout -subject -dates

# I. サーバーのCSRに署名してserver.crtを発行する

> ⏳ **サーバー役を待とう！**
>
> サーバー役が `server.csr` を作成して送ってくるまでここで待機だよ。
> `-----BEGIN CERTIFICATE REQUEST-----` ～ `-----END CERTIFICATE REQUEST-----` が届いたら次に進もう！

## 1.1 CSRを受け取ろう！

サーバー役から送られてきた CSR を保存するよ。

`-----BEGIN CERTIFICATE REQUEST-----` から `-----END CERTIFICATE REQUEST-----` までを丸ごと貼り付けてね！

In [ ]:
# @title CSRを保存する {"run":"auto"}
server_csr = "" # @param {"type":"string","placeholder":"-----BEGIN CERTIFICATE REQUEST----- ... -----END CERTIFICATE REQUEST-----"}

# csr形式に成形
server_csr = server_csr.replace(" ", "\n")
server_csr = server_csr.replace("-----BEGIN\nCERTIFICATE\nREQUEST-----", "-----BEGIN CERTIFICATE REQUEST-----")
server_csr = server_csr.replace("-----END\nCERTIFICATE\nREQUEST-----", "-----END CERTIFICATE REQUEST-----")
open("server.csr", "w").write(server_csr)
print(server_csr)
print("保存完了")

In [ ]:
# CSR の内容を確認しよう（CN が TLS-Theater-Server になっていれば OK！）
!openssl req -in server.csr -noout -subject

## 1.2 CSRに署名してserver.crtを発行しよう！

`ca.key` で署名することで「このサーバーはCAが認めた正規のサーバーである」という証明書が完成するよ。

これが実世界の証明書発行プロセスそのものだ！

In [ ]:
!openssl x509 -req -days 1 \
  -in     server.csr \
  -CA     ca.crt \
  -CAkey  ca.key \
  -CAcreateserial \
  -out    server.crt

In [ ]:
# 署名が通っているか確認しよう！
# → server.crt: OK と出ればOK！
!openssl verify -CAfile ca.crt server.crt

In [ ]:
# フィンガープリントも確認しておこう（参加者への確認用）
!openssl x509 -in server.crt -fingerprint -sha256 -noout

## 1.3 server.crtをサーバー役に返そう！

署名済みの証明書をサーバー役に渡そう。Discord の DM などで送ってあげてね。

セッション当日に Discord に貼るのはサーバー役の仕事だよ！

In [ ]:
!cat server.crt

上の出力をコピーして、サーバー役に DM しよう！

```
署名しました。これがあなたの証明書です。
当日の Certificate ステップでこれを Discord に貼ってください。

-----BEGIN CERTIFICATE-----
（cat の出力をそのまま貼る）
-----END CERTIFICATE-----
```

# II. ca.crtを全参加者に配布する

## 2.1 ca.crtを確認して全員に配布しよう！

`ca.crt` は秘密情報じゃないよ！これはブラウザや OS に組み込まれた **ルートCA証明書ストア** に相当するもの。

セッション前に Discord の全員が見えるチャンネルに貼ってOKだよ。クライアント役はこれを使ってサーバー証明書を検証するよ。

> 💡 **観客への解説ポイント**：実際のブラウザには数百のルートCA証明書が最初から組み込まれているよ。
> 今日の `ca.crt` はそのうちの1つに相当するんだ。
> これを信頼するということは「このCAが署名した証明書は全部信頼する」ということだよ！

In [ ]:
!cat ca.crt

上の出力をコピーして、全員が見えるチャンネルに投稿しよう！

```
【事前配布】今日のセッションで使うルートCA証明書です。
クライアント役の方はこれを ca.crt として保存してください。
ブラウザの組み込みルートCAに相当します（ca.key は私だけが持っています）。

-----BEGIN CERTIFICATE-----
（cat の出力をそのまま貼る）
-----END CERTIFICATE-----
```

# III. セッション当日（MC・進行役として）

CA の事前作業はここで完了だよ！お疲れさま 🎉

当日はサーバー役・クライアント役がハンドシェイクを進めるから、あなたは MC として各ステップの意味を観客に解説する役割に徹しよう。

---

**Certificate ステップでの解説例：**

「サーバーが証明書を送りました。クライアントは今、この証明書が本当に私（CA）の署名を持っているかを `openssl verify` で確認しています。ブラウザが錠前マークを表示するのはまさにこの検証が通ったときです」

---

# 完了！

おつかれさま！！

パケットはみんなに見えていたのに、内容はバレなかったよね。

これがTLSだよ！